In [1]:
from pystac_client import Client
import joblib
from typing import Any
import os
import requests
from loguru import logger
from tqdm import tqdm
from datetime import datetime, timezone

In [2]:
from utils.constants import GHANA_GDF, SENTINEL_SCENES_FOLDERPATH, SILVER_FOLDERPATH, EARCH_SEARCH_API_URL, SENTINEL_BANDS
from utils.site import Site

In [3]:
# Load stac_items
stac_items = joblib.load(SILVER_FOLDERPATH / "stac_items.joblib")
bboxes = list(set([tuple(e.bbox) for e in stac_items]))
print(f"Loaded {len(stac_items)} STAC items, covering {len(bboxes)} unique bboxes.")

Loaded 48 STAC items, covering 48 unique bboxes.


In [4]:
# Download these STAC items
def download_file(url: str, filepath: Path) -> None:
    """Downloads a file using a .part extension to prevent corruption if interrupted."""
    if filepath.exists():
        return

    part_filepath = filepath.with_suffix(filepath.suffix + ".part")

    response = requests.get(url, stream=True)
    response.raise_for_status()

    total_size = int(response.headers.get("content-length", 0))

    with part_filepath.open("wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                size = f.write(chunk)

    part_filepath.rename(filepath)

for si in tqdm(stac_items, total=len(stac_items)):
    print(si.id)
    # Prep the filesystem
    scene_folder = SENTINEL_SCENES_FOLDERPATH / si.id
    scene_folder.mkdir(parents=True, exist_ok=True)

    for band in SENTINEL_BANDS:
        if band in si.assets:
            download_url = si.assets[band].href
            filename = f"{si.id}_{band}.tif"
            filepath = scene_folder / filename

            try:
                download_file(download_url, filepath)
            except Exception as e:
                logger.exception(f"Error downloading {band}: {e}")

    joblib.dump(si, scene_folder / 'stac_item.joblib')

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 48/48 [00:00<00:00, 261.84it/s]

S2B_30NXM_20200107_0_L2A
S2B_30NYM_20200127_0_L2A
S2A_30NYM_20200122_0_L2A
S2B_30NYM_20200117_0_L2A
S2A_30NYM_20200112_0_L2A
S2B_30NYM_20200107_0_L2A
S2A_30NYM_20200102_0_L2A
S2B_30NWN_20200127_0_L2A
S2B_30NWP_20200127_0_L2A
S2A_30NWN_20200122_0_L2A
S2B_30NWN_20200117_0_L2A
S2B_30NWP_20200117_0_L2A
S2A_30NWN_20200112_0_L2A
S2A_30NWP_20200112_0_L2A
S2B_30NWN_20200107_0_L2A
S2B_30NWP_20200107_0_L2A
S2A_30NWN_20200102_0_L2A
S2A_30NWP_20200102_0_L2A
S2B_30NWM_20200127_0_L2A
S2A_30NWM_20200122_0_L2A
S2A_30NWM_20200112_0_L2A
S2B_30NWM_20200107_0_L2A
S2A_30NWM_20200102_0_L2A
S2B_30NWM_20200130_0_L2A
S2A_30NWM_20200125_0_L2A
S2A_30NWM_20200105_0_L2A
S2A_30NVM_20200105_0_L2A
S2B_30NXL_20200107_0_L2A
S2B_30NYL_20200127_0_L2A
S2B_30NYL_20200117_0_L2A
S2A_30NYL_20200112_0_L2A
S2B_30NYL_20200107_0_L2A
S2A_30NYL_20200102_0_L2A
S2B_30NWN_20200130_0_L2A
S2A_30NWN_20200125_0_L2A
S2B_30NWN_20200120_0_L2A
S2A_30NWN_20200115_0_L2A
S2B_30NWN_20200110_0_L2A
S2A_30NWN_20200105_0_L2A
S2B_30NWP_20200130_0_L2A
